In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

# 1. LOAD & ENHANCED FEATURE ENGINEERING
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

physical_stats = ['Sprint_40yd', 'Vertical_Jump', 'Weight', 'Height', 'Broad_Jump']

for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    # Rank-based features often help AUC
    for stat in physical_stats:
        df[f'{stat}_pos_rel'] = df[stat] - df.groupby('Position')[stat].transform('mean')
        df[f'{stat}_rank'] = df.groupby('Position')[stat].rank(pct=True)

school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)

drop_cols = ["Id", "School"]
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

X = train.drop(columns=["Drafted"])
y = train["Drafted"]

imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test), columns=test.columns)

scaler = RobustScaler()
X_imputed = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns)
test_imputed = pd.DataFrame(scaler.transform(test_imputed), columns=test.columns)

# 2. GENERATE BASE MODEL OOF PREDICTIONS
cat_params = {'iterations': 1000, 'depth': 6, 'learning_rate': 0.02, 'random_seed': 42, 'verbose': 0}
xgb_params = {'n_estimators': 800, 'max_depth': 5, 'learning_rate': 0.02, 'random_state': 42, 'eval_metric': 'logloss'}
lgbm_params = {'n_estimators': 800, 'learning_rate': 0.02, 'random_state': 42, 'objective': 'binary', 'verbosity': -1}

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
oof_preds = np.zeros((len(X_imputed), 3))
test_preds = np.zeros((len(test_imputed), 3))

models = [
    CatBoostClassifier(**cat_params),
    XGBClassifier(**xgb_params),
    LGBMClassifier(**lgbm_params)
]

for i, model in enumerate(models):
    for train_idx, valid_idx in rskf.split(X_imputed, y):
        Xt, Xv = X_imputed.iloc[train_idx], X_imputed.iloc[valid_idx]
        yt, yv = y.iloc[train_idx], y.iloc[valid_idx]
        model.fit(Xt, yt)
        oof_preds[valid_idx, i] += model.predict_proba(Xv)[:, 1] / 2
        test_preds[:, i] += model.predict_proba(test_imputed)[:, 1] / 10

# 3. PYTORCH META-LEARNER
class MetaModel(nn.Module):
    def __init__(self, input_dim):
        super(MetaModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

# Convert to tensors
X_meta_train = torch.FloatTensor(oof_preds)
y_meta_train = torch.FloatTensor(y.values).view(-1, 1)
X_meta_test = torch.FloatTensor(test_preds)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MetaModel(input_dim=3).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# Training loop
model.train()
for epoch in range(150):
    optimizer.zero_grad()
    outputs = model(X_meta_train.to(device))
    loss = criterion(outputs, y_meta_train.to(device))
    loss.backward()
    optimizer.step()

# 4. FINAL INFERENCE
model.eval()
with torch.no_grad():
    final_predictions = model(X_meta_test.to(device)).cpu().numpy().flatten()

submission = sample_sub.copy()
submission["Drafted"] = final_predictions
submission.to_csv("submission_pytorch_stacking.csv", index=False)
print("Submission generated with PyTorch meta-learner.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22940\3581121353.py:97: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_meta_train = torch.FloatTensor(y.values).view(-1, 1)


Submission generated with PyTorch meta-learner.
